# Chapter 14 — Computational Pathology Lab
## IDC Patch Classification with Transfer Learning (ResNet-50)

> **Google Colab** — Run on a T4 GPU (Runtime → Change runtime type → T4 GPU).
> The only manual step required is uploading your `kaggle.json` credentials file when prompted in Section 3.

---

### 1.1 Clinical Motivation

Breast cancer is the most commonly diagnosed cancer in women worldwide, accounting for roughly 2.3 million new cases per year (WHO, 2020). The gold-standard diagnostic workflow relies on **haematoxylin and eosin (H&E) staining** of formalin-fixed paraffin-embedded (FFPE) tissue sections. A pathologist examines these stained slides under a microscope and visually assesses nuclear pleomorphism, mitotic figures, glandular architecture, and stromal infiltration.

The most common subtype of invasive breast cancer is **Invasive Ductal Carcinoma (IDC)**, which begins in the milk ducts and invades surrounding tissue. Accurate identification of IDC-positive regions in whole-slide images (WSIs) is crucial for:

- Grading tumour aggressiveness (Nottingham grading system)
- Determining surgical margins
- Guiding adjuvant therapy decisions

Manual slide review is time-consuming, subject to inter-observer variability, and increasingly bottlenecked by the global shortage of trained pathologists. **Computational pathology** — applying deep learning to digitised slides — offers a scalable, reproducible complement to expert review.

### 1.2 The Computational Task: IDC Patch Classification

In this lab we train a binary convolutional neural network classifier to distinguish:

| Label | Meaning |
|-------|---------|
| **0** | IDC-negative patch (no invasive ductal carcinoma tissue present) |
| **1** | IDC-positive patch (contains invasive ductal carcinoma tissue) |

Each input is a **50 × 50 pixel RGB image patch** extracted from a TCGA whole-mount H&E slide. We upscale to 224 × 224 px to match the input expected by ImageNet-pretrained models — a valid approach for transfer-learning demonstrations (see Section 5).

### 1.3 Dataset: Kaggle Breast Histopathology Images

We use the publicly available **paultimothymooney/breast-histopathology-images** dataset on Kaggle:

- **Source**: 162 TCGA whole-mount H&E breast cancer slide scans digitised at 40× magnification
- **Patches**: 277,524 image patches of 50 × 50 pixels
- **Labels**: IDC-negative (198,738 patches, ~72%) and IDC-positive (78,786 patches, ~28%)
- **Format**: PNG files organised on disk as `<patient_id>/<label>/<patient_id>_idx5_x<X>_y<Y>_class<0|1>.png`
- **Access**: Public Kaggle dataset — no dbGaP approval or GDC data access agreement required

### 1.4 Pipeline Overview

| Step | Description |
|------|-------------|
| **1. Download** | Kaggle CLI downloads and unzips the dataset (~1.6 GB) |
| **2. Inventory** | Walk directory tree → `tiles_df` (path, label, patient_id) |
| **3. Split** | Patient-level stratified 70/15/15 train/val/test split |
| **4. Augment** | H&E-aware augmentation (flips, rotations, colour jitter) |
| **5. Dataset** | `PathologyTileDataset` — returns (tensor, label, patient_id) |
| **6. Sample** | `WeightedRandomSampler` to handle class imbalance |
| **7. Model** | ResNet-50 + custom head (frozen backbone Stage 1, full fine-tune Stage 2) |
| **8. Train** | CrossEntropyLoss, AdamW, CosineAnnealingWarmRestarts |
| **9. Evaluate** | AUC, F1, confusion matrix, ROC curve, patient-level soft voting |

> **Note on dbGaP / WSIs**: This notebook intentionally bypasses the raw TCGA SVS whole-slide images, which require a dbGaP data access agreement, a GDC API token, and OpenSlide for tissue detection and tiling. The Kaggle dataset used here provides pre-extracted patches from the same TCGA cohort, making the pipeline accessible to anyone with a free Kaggle account.

---
## Section 2 — Imports and Reproducibility

In [ ]:
# Install additional dependencies not pre-installed in Colab
!pip install kaggle -q

In [ ]:
import os
import random
import shutil
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Section 3 — Dataset: Kaggle Breast Histopathology Images

### 3.1 Kaggle API Credentials

To download the dataset you need a **Kaggle account** (free) and an API token:

1. Log in to [kaggle.com](https://www.kaggle.com) → Profile → Settings → API → **Create New Token**
2. This downloads `kaggle.json` to your local machine
3. Run the cell below and upload `kaggle.json` when the file chooser appears

> **Privacy note**: `kaggle.json` contains your Kaggle username and API key. It is only stored in the Colab VM's memory and is deleted when the runtime is disconnected.

In [ ]:
from google.colab import files

print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Place credentials where the Kaggle CLI expects them
kaggle_dir = Path(os.path.expanduser("~/.kaggle"))
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"

with open(kaggle_json_path, "wb") as f:
    f.write(uploaded["kaggle.json"])

# Restrict permissions (required by Kaggle CLI)
os.chmod(kaggle_json_path, 0o600)
print(f"Credentials saved to {kaggle_json_path}")

### 3.2 Download the Dataset

The dataset is approximately **1.6 GB** after unzipping. Download typically takes 3–5 minutes on a Colab T4 instance.

In [ ]:
DATA_DIR = Path("/content/idc_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d paultimothymooney/breast-histopathology-images \
    --unzip -p /content/idc_data -q

print("Download complete.")
print(f"Contents of {DATA_DIR}:")
!ls /content/idc_data | head -10

### 3.3 On-Disk Structure

After unzipping, the dataset follows this directory layout:

```
/content/idc_data/
  └── <patient_id>/          # e.g. 8863
        ├── 0/               # IDC-negative patches
        │     └── 8863_idx5_x1201_y1251_class0.png
        └── 1/               # IDC-positive patches
              └── 8863_idx5_x1051_y1051_class1.png
```

The filename encodes the **patient ID**, patch index, spatial coordinates (x, y) within the original WSI, and the class label:

```
<patientId>_idx5_x<X>_y<Y>_class<0|1>.png
```

We parse `patient_id` from the numeric prefix before `_idx5`.

### 3.4 Build the Patch Inventory DataFrame

In [ ]:
def build_tiles_df(data_dir: Path) -> pd.DataFrame:
    """
    Walk the IDC dataset directory tree and return a DataFrame with columns:
      - tile_path  : absolute path to the PNG patch
      - label      : int, 0 = IDC-negative, 1 = IDC-positive
      - patient_id : str, numeric patient identifier parsed from filename
    """
    records = []
    # Kaggle dataset may have an extra nested folder; find the actual root
    # by locating directories that contain '0' and '1' subdirs
    for png_path in data_dir.rglob("*.png"):
        # Expected filename: <patientId>_idx5_x<X>_y<Y>_class<0|1>.png
        stem = png_path.stem  # e.g. "8863_idx5_x1201_y1251_class0"
        if "_idx5_" not in stem:
            continue
        patient_id = stem.split("_idx5_")[0]  # e.g. "8863"
        label = int(png_path.parent.name)      # parent dir is "0" or "1"
        records.append({
            "tile_path": str(png_path),
            "label": label,
            "patient_id": patient_id,
        })
    df = pd.DataFrame(records)
    return df


print("Building patch inventory (this may take ~1-2 minutes for 277k files) …")
tiles_df = build_tiles_df(DATA_DIR)
print(f"Total patches: {len(tiles_df):,}")
print(f"Unique patients: {tiles_df['patient_id'].nunique()}")
print("\nLabel distribution:")
counts = tiles_df["label"].value_counts().sort_index()
for lbl, cnt in counts.items():
    name = "IDC-negative" if lbl == 0 else "IDC-positive"
    print(f"  {lbl} ({name}): {cnt:,}  ({100 * cnt / len(tiles_df):.1f}%)")

---
## Section 4 — Patient-Level Data Splits

### Why patient-level splitting matters

The most common source of **data leakage** in computational pathology is splitting data at the patch level rather than the patient level. Consider what happens if patches from the same patient appear in both the training and test sets:

- Patches from the same slide share global slide-level characteristics (staining intensity, scanner artefacts, tissue processing batch effects)
- A model trained on some patches from patient *P* will have "seen" the staining style of that patient and will trivially classify unseen patches from the same patient correctly — not because it has learned IDC morphology, but because it has memorised slide-level confounders
- The resulting test AUC will be inflated and will not generalise to new patients

This dataset is well-suited for demonstrating the correct approach: **each patch filename encodes the patient ID**, so we can enforce a strict patient-level split. No patches from any test patient appear in training or validation.

We use a **70 / 15 / 15** stratified split on `patient_id`, stratifying by the per-patient IDC prevalence (proportion of positive patches), then propagate the assignment to all patches of each patient.

In [ ]:
# ── Patient-level stratification ─────────────────────────────────────────────
# Compute per-patient label (majority class) for stratification
patient_stats = (
    tiles_df.groupby("patient_id")["label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "idc_frac", "count": "n_patches"})
    .reset_index()
)
# Bin IDC fraction into categories for stratification
patient_stats["strat_bin"] = pd.cut(
    patient_stats["idc_frac"], bins=[-0.001, 0.0, 0.2, 0.5, 1.001],
    labels=["pure_neg", "low_idc", "mid_idc", "high_idc"]
)

patients = patient_stats["patient_id"].values
strat_labels = patient_stats["strat_bin"].cat.codes.values

# Split: 85% train+val / 15% test
splitter1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trainval_idx, test_idx = next(splitter1.split(patients, strat_labels, groups=patients))
trainval_patients = patients[trainval_idx]
test_patients = set(patients[test_idx])

# Split: 70/15 train/val from the 85%
val_fraction_of_trainval = 0.15 / 0.85
strat_tv = strat_labels[trainval_idx]
splitter2 = GroupShuffleSplit(n_splits=1, test_size=val_fraction_of_trainval, random_state=SEED)
train_idx_local, val_idx_local = next(
    splitter2.split(trainval_patients, strat_tv, groups=trainval_patients)
)
train_patients = set(trainval_patients[train_idx_local])
val_patients = set(trainval_patients[val_idx_local])

# Propagate patient assignments to patch level
def assign_split(pid):
    if pid in train_patients:   return "train"
    if pid in val_patients:     return "val"
    return "test"

tiles_df["split"] = tiles_df["patient_id"].map(assign_split)

# ── Print split statistics ───────────────────────────────────────────────────
print("Patient-level split:")
print(f"  Train patients : {len(train_patients)}")
print(f"  Val   patients : {len(val_patients)}")
print(f"  Test  patients : {len(test_patients)}")
print()
print("Patch-level split:")
for split_name in ["train", "val", "test"]:
    sub = tiles_df[tiles_df["split"] == split_name]
    pos = (sub["label"] == 1).sum()
    print(f"  {split_name:5s}: {len(sub):>7,} patches  "
          f"({pos:>6,} IDC-pos, {len(sub)-pos:>7,} IDC-neg)  "
          f"IDC+ rate: {100*pos/len(sub):.1f}%")

---
## Section 5 — Image Preprocessing and H&E-Aware Augmentation

### 5.1 Augmentation Strategy

H&E-stained tissue sections have specific symmetry properties that inform our choice of augmentations:

| Augmentation | Rationale |
|--------------|-----------|
| Horizontal & Vertical flip | Tissue orientation is arbitrary; IDC morphology is isotropic |
| 90° rotation | No preferred orientation in H&E slides |
| Colour jitter (brightness, contrast, saturation, hue) | Staining intensity varies across labs, batches, and slide scanners |
| Random horizontal flip | Additional left-right augmentation |

We do **not** use aggressive geometric distortions (e.g., elastic deformation) or extreme colour shifts that would destroy nuclear morphology cues.

### 5.2 Upscaling 50 × 50 → 224 × 224

> **Note on resolution**: The Kaggle patches are 50 × 50 pixels. We resize them to 224 × 224 px to match the input resolution expected by the ImageNet-pretrained ResNet-50. This upscaling introduces no new information — it simply replicates pixels via bilinear interpolation — but it allows us to reuse the pretrained convolutional feature hierarchy without architectural changes. In a production pipeline, you would tile the original WSI at a resolution that yields ~224 × 224 pixel patches at the target magnification (typically 20×), bypassing the upscaling step.

In [ ]:
IMG_SIZE = 224

# ── Transforms ───────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(degrees=90),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def denormalise(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


print("Transforms defined.")

In [ ]:
# ── Augmentation preview ─────────────────────────────────────────────────────
# Pick one IDC-positive and one IDC-negative patch from the training split
train_df = tiles_df[tiles_df["split"] == "train"].reset_index(drop=True)

example_paths = {
    "IDC-negative": train_df[train_df["label"] == 0].iloc[0]["tile_path"],
    "IDC-positive": train_df[train_df["label"] == 1].iloc[0]["tile_path"],
}

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle("Augmentation Preview: original + 5 augmented versions", fontsize=13)

for row_idx, (class_name, path) in enumerate(example_paths.items()):
    img = Image.open(path).convert("RGB")
    # Original (no augmentation, just resized for display)
    ax = axes[row_idx, 0]
    ax.imshow(img.resize((IMG_SIZE, IMG_SIZE)))
    ax.set_title(f"{class_name}\n(original)", fontsize=8)
    ax.axis("off")
    # Augmented versions using the exact training transform
    for col_idx in range(1, 6):
        tensor = train_transform(img)
        ax = axes[row_idx, col_idx]
        ax.imshow(denormalise(tensor).permute(1, 2, 0).numpy())
        ax.set_title(f"aug {col_idx}", fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()


---
## Section 6 — Custom PyTorch Dataset

We implement `PathologyTileDataset`, which:
- Reads a PNG patch from disk and converts to RGB
- Applies the specified transform (augmentation for train, resize+normalise for val/test)
- Returns a 3-tuple `(image_tensor, label, patient_id)` — the `patient_id` string is needed for patient-level aggregation in Section 15

In [ ]:
class PathologyTileDataset(Dataset):
    """
    PyTorch Dataset for the IDC breast histopathology patch dataset.

    Label semantics:
        0 = IDC-negative  (no invasive ductal carcinoma tissue)
        1 = IDC-positive  (invasive ductal carcinoma tissue present)

    Returns
    -------
    (image_tensor, label, patient_id)
        image_tensor : FloatTensor of shape (3, H, W), ImageNet-normalised
        label        : int
        patient_id   : str  (used for patient-level aggregation)
    """

    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["tile_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"]), str(row["patient_id"])


print("PathologyTileDataset defined.")

---
## Section 7 — DataLoaders and Balanced Sampling

### Class Imbalance

The IDC dataset has a **~72% / 28% class imbalance** (IDC-negative majority):

- A naive classifier that always predicts "IDC-negative" achieves 72% accuracy — not useful clinically
- A false negative (missed IDC) has severe clinical consequences; we want high recall for IDC-positive

We address imbalance with **`WeightedRandomSampler`**, which oversamples the minority class (IDC-positive) during training so that each mini-batch is approximately class-balanced. This is equivalent to class-weighted loss but operates at the data sampling level.

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2

# ── Datasets ─────────────────────────────────────────────────────────────────
train_df_split = tiles_df[tiles_df["split"] == "train"].reset_index(drop=True)
val_df_split   = tiles_df[tiles_df["split"] == "val"].reset_index(drop=True)
test_df_split  = tiles_df[tiles_df["split"] == "test"].reset_index(drop=True)

train_dataset = PathologyTileDataset(train_df_split, transform=train_transform)
val_dataset   = PathologyTileDataset(val_df_split,   transform=val_transform)
test_dataset  = PathologyTileDataset(test_df_split,  transform=val_transform)

# ── WeightedRandomSampler ─────────────────────────────────────────────────────
train_labels = train_df_split["label"].values
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

# ── Custom collate_fn ─────────────────────────────────────────────────────────
# The default collate_fn cannot handle mixed-type tuples (tensor, int, str)
def collate_fn(batch):
    images, labels, patient_ids = zip(*batch)
    return torch.stack(images), torch.tensor(labels, dtype=torch.long), list(patient_ids)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True,
)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

---
## Section 8 — Model Architecture: Two-Stage Transfer Learning

### Strategy

We use **ResNet-50** pretrained on ImageNet as a feature extractor, replacing the final fully connected layer with a custom classification head tailored for our binary IDC task.

Transfer learning proceeds in two stages:

| Stage | Backbone | Learning rate | Purpose |
|-------|----------|---------------|---------|
| **1** | Frozen | 1 × 10⁻³ | Train only the new head; prevents destroying pretrained features with high gradients |
| **2** | Unfrozen | 1 × 10⁻⁴ | Fine-tune all layers end-to-end at a low LR; adapts ImageNet features to H&E histology |

The custom head uses **dropout** for regularisation and a **ReLU** nonlinearity before the output logits. The output is a 2-dimensional logit vector `[score_IDC-neg, score_IDC-pos]`.

In [ ]:
def build_model(num_classes: int = 2, dropout: float = 0.4) -> nn.Module:
    """
    ResNet-50 with a custom classification head.

    Output logits: [score_IDC-neg, score_IDC-pos]
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features  # 2048
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(512, num_classes),
    )
    return model


def freeze_backbone(model: nn.Module):
    """Freeze all layers except the custom head (model.fc)."""
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("fc.")


def unfreeze_backbone(model: nn.Module):
    """Unfreeze all model parameters for end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True


model = build_model().to(DEVICE)
print(model)

total_params    = sum(p.numel() for p in model.parameters())
trainable_init  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters      : {total_params:,}")
print(f"Trainable (all unfrozen): {trainable_init:,}")

---
## Section 9 — Loss Function, Optimiser, and LR Schedule

| Component | Choice | Rationale |
|-----------|--------|----------|
| **Loss** | `CrossEntropyLoss` | Standard multi-class loss; compatible with balanced sampling |
| **Optimiser** | `AdamW` | Adam with decoupled weight decay; strong default for vision fine-tuning |
| **LR schedule** | `CosineAnnealingWarmRestarts` | Periodic LR restarts help escape sharp minima; especially beneficial in Stage 2 when fine-tuning the full backbone |

In [ ]:
STAGE1_LR     = 1e-3
STAGE2_LR     = 1e-4
WEIGHT_DECAY  = 1e-4
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 10

criterion = nn.CrossEntropyLoss()


def make_optimiser_and_scheduler(model, lr, total_epochs):
    optimiser = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimiser, T_0=max(1, total_epochs // 2), T_mult=1
    )
    return optimiser, scheduler


print("Loss, optimiser, and scheduler factory defined.")

---
## Section 10 — Training and Evaluation Helper Functions

In [ ]:
def train_one_epoch(model, loader, optimiser, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        optimiser.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
    scheduler.step()
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        probs = torch.softmax(logits, dim=1)[:, 1]  # IDC-positive probability
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / total, correct / total, auc


print("Training and evaluation helpers defined.")

---
## Section 11 — Training Loop with Two-Stage Fine-Tuning

In [ ]:
history = {
    "train_loss": [], "train_acc": [],
    "val_loss":   [], "val_acc":   [], "val_auc": [],
    "stage_boundary": STAGE1_EPOCHS,
}

best_val_auc = 0.0
best_model_state = None

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Stage 1 — Frozen backbone, train head only
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("="*60)
print("STAGE 1: Frozen backbone — training head only")
print("="*60)
freeze_backbone(model)
optimiser, scheduler = make_optimiser_and_scheduler(model, STAGE1_LR, STAGE1_EPOCHS)

for epoch in range(1, STAGE1_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimiser, scheduler, criterion, DEVICE)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion, DEVICE)
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"  Epoch {epoch:2d}/{STAGE1_EPOCHS} | "
          f"train loss {tr_loss:.4f}  acc {tr_acc:.3f} | "
          f"val loss {val_loss:.4f}  acc {val_acc:.3f}  AUC {val_auc:.4f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Stage 2 — Unfreeze backbone, end-to-end fine-tuning
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*60)
print("STAGE 2: Unfrozen backbone — end-to-end fine-tuning")
print("="*60)
unfreeze_backbone(model)
optimiser, scheduler = make_optimiser_and_scheduler(model, STAGE2_LR, STAGE2_EPOCHS)

for epoch in range(1, STAGE2_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimiser, scheduler, criterion, DEVICE)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion, DEVICE)
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"  Epoch {epoch:2d}/{STAGE2_EPOCHS} | "
          f"train loss {tr_loss:.4f}  acc {tr_acc:.3f} | "
          f"val loss {val_loss:.4f}  acc {val_acc:.3f}  AUC {val_auc:.4f}")

# Restore best checkpoint
model.load_state_dict(best_model_state)
print(f"\nBest validation AUC: {best_val_auc:.4f}  — checkpoint restored.")

---
## Section 12 — Training Dynamics Plot

In [ ]:
total_epochs = STAGE1_EPOCHS + STAGE2_EPOCHS
epochs_range = range(1, total_epochs + 1)
boundary = STAGE1_EPOCHS + 0.5  # vertical line between stages

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Training Dynamics", fontsize=14)

plot_configs = [
    ("train_loss", "val_loss",  "Loss",     "Loss"),
    ("train_acc",  "val_acc",   "Accuracy", "Accuracy"),
    (None,         "val_auc",   "Val AUC",  "AUC"),
]

for ax, (train_key, val_key, title, ylabel) in zip(axes, plot_configs):
    if train_key:
        ax.plot(epochs_range, history[train_key], label="Train", color="steelblue")
    ax.plot(epochs_range, history[val_key], label="Val", color="darkorange")
    ax.axvline(x=boundary, color="gray", linestyle="--", linewidth=1.5)
    ax.text(boundary + 0.2, ax.get_ylim()[0], "Stage 2", fontsize=8, color="gray")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 13 — Patch-Level Evaluation on Held-Out Test Set

We evaluate the best checkpoint on the held-out test set (patients the model has never seen during training or validation). We report:

- **Accuracy** — overall fraction of correctly classified patches
- **AUC** — area under the ROC curve; threshold-independent measure of discrimination
- **Classification report** — per-class precision, recall, and F1 score
- **Confusion matrix** — absolute counts of TP, TN, FP, FN

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    """Run inference; return arrays of true labels, predicted labels, IDC+ probs, and patient IDs."""
    model.eval()
    all_labels, all_preds, all_probs, all_pids = [], [], [], []
    for images, labels, patient_ids in loader:
        images = images.to(device)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()  # IDC+ prob
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_pids.extend(patient_ids)
    return (
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
        np.array(all_pids),
    )


test_labels, test_preds, test_probs, test_pids = predict(model, test_loader, DEVICE)

test_acc = (test_labels == test_preds).mean()
test_auc = roc_auc_score(test_labels, test_probs)

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test AUC      : {test_auc:.4f}")
print()
print(classification_report(
    test_labels, test_preds,
    target_names=["IDC-neg (0)", "IDC-pos (1)"]
))

---
## Section 14 — Confusion Matrix and ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["IDC-negative", "IDC-positive"]
)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — Patch Level")

# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(test_labels, test_probs)
axes[1].plot(fpr, tpr, color="darkorange", lw=2,
             label=f"ROC (AUC = {test_auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--",
             label="Random classifier")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — IDC-positive vs IDC-negative")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 15 — Patient-Level Aggregation by Soft Voting

### Motivation

Patch-level predictions are useful for localisation, but the **clinically relevant question** is: *does this patient have IDC?* To answer this, we aggregate the patch-level IDC-positive probabilities across all patches from each patient using **soft voting** (also called probability averaging):

$$\hat{p}_{\text{IDC+}}^{(i)} = \frac{1}{N_i} \sum_{j=1}^{N_i} p_{\text{IDC+}}^{(i,j)}$$

where $N_i$ is the number of patches from patient $i$ and $p_{\text{IDC+}}^{(i,j)}$ is the model's IDC-positive probability for patch $j$ of patient $i$.

The final patient-level prediction is $\hat{y}_i = \mathbf{1}[\hat{p}_{\text{IDC+}}^{(i)} > 0.5]$.

The patient-level ground truth is 1 if **any** patch from that patient is IDC-positive (i.e., the patient has IDC), and 0 otherwise.

In [ ]:
# ── Patient-level aggregation ─────────────────────────────────────────────────
patient_data = defaultdict(lambda: {"probs": [], "labels": []})
for pid, prob, label in zip(test_pids, test_probs, test_labels):
    patient_data[pid]["probs"].append(prob)
    patient_data[pid]["labels"].append(label)

patient_results = []
for pid, data in patient_data.items():
    mean_prob  = np.mean(data["probs"])
    true_label = int(any(l == 1 for l in data["labels"]))  # 1 if patient has any IDC+ patch
    pred_label = int(mean_prob > 0.5)
    patient_results.append({
        "patient_id":  pid,
        "mean_idc_prob": mean_prob,
        "true_label":   true_label,
        "pred_label":   pred_label,
        "n_patches":    len(data["probs"]),
    })

patient_df = pd.DataFrame(patient_results)

# ── Patient-level metrics ─────────────────────────────────────────────────────
pat_acc = (patient_df["true_label"] == patient_df["pred_label"]).mean()
pat_auc = roc_auc_score(patient_df["true_label"], patient_df["mean_idc_prob"])

print(f"Patient-level accuracy : {pat_acc:.4f}")
print(f"Patient-level AUC      : {pat_auc:.4f}")
print()
print(classification_report(
    patient_df["true_label"],
    patient_df["pred_label"],
    target_names=["IDC-negative", "IDC-positive"]
))
print(patient_df.sort_values("mean_idc_prob", ascending=False).head(10).to_string(index=False))

---
## Section 16 — Qualitative Inspection

Visualising correctly and incorrectly classified patches helps identify systematic failure modes (e.g., mis-classifying necrotic tissue, stroma, or artefacts as IDC-positive).

In [ ]:
CLASS_NAMES = ["IDC-neg", "IDC-pos"]

# Rebuild index arrays
correct_mask   = test_labels == test_preds
incorrect_mask = ~correct_mask

correct_idx   = np.where(correct_mask)[0]
incorrect_idx = np.where(incorrect_mask)[0]

rng = np.random.default_rng(SEED)
n_show = min(8, len(correct_idx), len(incorrect_idx))
sample_correct   = rng.choice(correct_idx,   size=n_show, replace=False)
sample_incorrect = rng.choice(incorrect_idx, size=n_show, replace=False)

fig, axes = plt.subplots(2, n_show, figsize=(2.2 * n_show, 5))
fig.suptitle("Top row: Correct predictions  |  Bottom row: Incorrect predictions",
             fontsize=11)

for col, idx in enumerate(sample_correct):
    img = Image.open(test_df_split.iloc[idx]["tile_path"]).convert("RGB")
    ax = axes[0, col]
    ax.imshow(img)
    ax.set_title(
        f"True: {CLASS_NAMES[test_labels[idx]]}\n"
        f"Pred: {CLASS_NAMES[test_preds[idx]]}",
        fontsize=7, color="green"
    )
    ax.axis("off")

for col, idx in enumerate(sample_incorrect):
    img = Image.open(test_df_split.iloc[idx]["tile_path"]).convert("RGB")
    ax = axes[1, col]
    ax.imshow(img)
    ax.set_title(
        f"True: {CLASS_NAMES[test_labels[idx]]}\n"
        f"Pred: {CLASS_NAMES[test_preds[idx]]}",
        fontsize=7, color="red"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## Section 17 — Suggested Extensions

This lab demonstrates the core computational pathology pipeline using accessible pre-extracted patches. Here are natural next steps to deepen your understanding:

1. **Threshold optimisation**: The default classification threshold of 0.5 optimises accuracy. For clinical applications, adjust the threshold on the validation set to maximise sensitivity (recall for IDC-positive) at a fixed specificity, or use Youden's J statistic.

2. **Grad-CAM visualisation**: Apply Gradient-weighted Class Activation Mapping (Grad-CAM) to generate heatmaps showing which spatial regions of each 50 × 50 patch most strongly activate the IDC-positive prediction. This provides a form of model interpretability.

3. **Multi-scale features**: Experiment with EfficientNet-B3 or ViT (Vision Transformer) backbones, which may capture multi-scale features more naturally than ResNet.

4. **Stain normalisation**: Apply Macenko or Vahadane stain normalisation to reduce batch effects before training. This often improves generalisation across different scanners and staining protocols.

5. **Multiple Instance Learning (MIL)**: Re-frame the problem as MIL, where the bag is the patient and the instances are patches. The model learns to predict patient-level labels without patch-level supervision, which is the setting when only slide-level diagnoses are available.

6. **Uncertainty quantification**: Apply Monte Carlo Dropout or Deep Ensembles to estimate predictive uncertainty per patch. High-uncertainty patches may correspond to ambiguous morphology that warrants pathologist review.

7. **Extended TCGA cohort**: Include other TCGA breast cancer subtypes (Invasive Lobular Carcinoma, mixed, etc.) as additional classes, extending the binary classifier to a multi-class or hierarchical architecture.

8. **Return to WSIs**: For production-grade models, return to the raw TCGA SVS whole-slide images (accessible via the GDC Data Portal with dbGaP approval) and apply proper Otsu tissue detection and multi-resolution tiling rather than the pre-extracted 50 × 50 patches used here. This allows you to control tiling strategy, magnification, and patch overlap, and to leverage the full spatial context of the slide.